<a href="https://www.kaggle.com/code/avikdas567/vesuvius-challenge-surface-detection?scriptVersionId=293527403" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# Vesuvius Challenge - Surface Detection
# ============================================================

import os
import zipfile
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import tifffile as tiff

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# Config
# ----------------------------
SLICE_RADIUS = 3          # 7 slices input
PATCH_SIZE = 128
MAX_PATCHES = 600         # controls runtime
EPOCHS = 2
BATCH_SIZE = 4
LR = 1e-3
THRESHOLD = 0.5

# ----------------------------
# Paths
# ----------------------------
INPUT_DIR = "/kaggle/input/vesuvius-challenge-surface-detection"
TRAIN_IMG_DIR = os.path.join(INPUT_DIR, "train_images")
TRAIN_LBL_DIR = os.path.join(INPUT_DIR, "train_labels")
TEST_IMG_DIR  = os.path.join(INPUT_DIR, "test_images")
OUTPUT_DIR = "/kaggle/working/preds"
ZIP_PATH = "/kaggle/working/submission.zip"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ----------------------------
# TIFF reader
# ----------------------------
def read_tiff_stack(path):
    imgs = []
    with Image.open(path) as img:
        i = 0
        while True:
            try:
                img.seek(i)
                imgs.append(np.array(img))
                i += 1
            except EOFError:
                break
    return np.stack(imgs, axis=0)

# ----------------------------
# Tiny 2.5D CNN
# ----------------------------
class TinyUNet(nn.Module):
    def __init__(self, in_ch=7):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
        )

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.dec(x)
        return x

# ----------------------------
# Training dataset (patches)
# ----------------------------
class PatchDataset(Dataset):
    def __init__(self, img_paths, lbl_paths):
        self.samples = []

        for ip, lp in zip(img_paths, lbl_paths):
            vol = read_tiff_stack(ip).astype(np.float32)
            lbl = read_tiff_stack(lp)

            vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-6)

            Z, Y, X = vol.shape
            for _ in range(50):  # few samples per volume
                z = random.randint(SLICE_RADIUS, Z - SLICE_RADIUS - 1)
                y = random.randint(0, Y - PATCH_SIZE - 1)
                x = random.randint(0, X - PATCH_SIZE - 1)

                img = vol[z-SLICE_RADIUS:z+SLICE_RADIUS+1, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                gt  = lbl[z, y:y+PATCH_SIZE, x:x+PATCH_SIZE]

                # ignore unlabeled
                mask = (gt != 2).astype(np.float32)
                gt = (gt == 1).astype(np.float32)

                self.samples.append((img, gt, mask))
                if len(self.samples) >= MAX_PATCHES:
                    return

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, gt, mask = self.samples[idx]
        return (
            torch.from_numpy(img),
            torch.from_numpy(gt),
            torch.from_numpy(mask),
        )

# ----------------------------
# Prepare training data
# ----------------------------
train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))

img_paths = []
lbl_paths = []

for i in train_df.id:
    ip = os.path.join(TRAIN_IMG_DIR, f"{i}.tif")
    lp = os.path.join(TRAIN_LBL_DIR, f"{i}.tif")
    if os.path.exists(ip) and os.path.exists(lp):
        img_paths.append(ip)
        lbl_paths.append(lp)

print(f"Using {len(img_paths)} labeled volumes for training")

dataset = PatchDataset(img_paths, lbl_paths)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ----------------------------
# Train
# ----------------------------
model = TinyUNet(in_ch=2*SLICE_RADIUS+1).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()

model.train()
for epoch in range(EPOCHS):
    for x, y, m in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        m = m.to(DEVICE)

        opt.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x).squeeze(1)
            loss = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
            loss = (loss * m).sum() / (m.sum() + 1e-6)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

    print(f"Epoch {epoch+1}/{EPOCHS} done")

# ----------------------------
# Inference
# ----------------------------
model.eval()
test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))

with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        vid = row["id"]
        vol = read_tiff_stack(os.path.join(TEST_IMG_DIR, f"{vid}.tif")).astype(np.float32)
        vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-6)

        Z, Y, X = vol.shape
        out = np.zeros((Z, Y, X), np.uint8)

        for z in range(SLICE_RADIUS, Z - SLICE_RADIUS):
            stack = torch.from_numpy(
                vol[z-SLICE_RADIUS:z+SLICE_RADIUS+1]
            ).unsqueeze(0).to(DEVICE)

            prob = torch.sigmoid(model(stack))[0,0].cpu().numpy()
            surf = prob > THRESHOLD
            
            for dz in (-1, 0, 1):
                zz = z + dz
                if 0 <= zz < Z:
                    out[zz][surf] = 1

        tiff.imwrite(os.path.join(OUTPUT_DIR, f"{vid}.tif"), out)

# ----------------------------
# Zip
# ----------------------------
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, f), f)

print("READY TO SUBMIT:", ZIP_PATH)

Using device: cuda
Using 786 labeled volumes for training


/tmp/ipykernel_24/1551753661.py:155: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_24/1551753661.py:165: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/2 done
Epoch 2/2 done


100%|██████████| 1/1 [00:02<00:00,  2.98s/it]


READY TO SUBMIT: /kaggle/working/submission.zip
